# Project-Alpha 复现笔记

复现招商证券《端到端的动态Alpha模型》(2023)

In [ ]:
import pandas as pd
import numpy as np
import torch

from config import *
from utils import fetch_stock_data, preprocess_pipeline, FactorDataset
from utils.metrics import rank_ic, calc_ic_series, ic_summary, group_return
from models import LinearAlphaModel, MLPAlphaModel
from losses import mse_loss, ic_loss, ccc_loss, orthogonal_penalty

## 1. 环境验证

In [ ]:
print(f"Python: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {DEVICE}")

## 2. 数据获取与预处理

In [ ]:
# 数据获取（首次运行需要几分钟）
from utils.data_loader import fetch_stock_data, compute_derived_factors

raw_df = fetch_stock_data()
raw_df = compute_derived_factors(raw_df)
print(f"原始数据: {raw_df.shape}")
raw_df.head()

In [ ]:
# 预处理
processed_df = preprocess_pipeline(raw_df)
print(f"预处理后: {processed_df.shape}")
processed_df.head()

## 3. 模型训练

In [ ]:
from train import train, run_training, plot_training_history
from utils.dataset import split_dataset

train_ds, val_ds, test_ds = split_dataset(processed_df)

### 3.1 线性基准模型

In [ ]:
n_factors = len([c for c in FACTOR_COLS if c in processed_df.columns])
linear_model = LinearAlphaModel(n_factors)
linear_history = train(linear_model, train_ds, val_ds, loss_name="mse", save_name="linear_mse")
plot_training_history(linear_history, title="Linear + MSE")

### 3.2 MLP模型

In [ ]:
mlp_model = MLPAlphaModel(n_factors)
mlp_history = train(mlp_model, train_ds, val_ds, loss_name="mse", is_mlp=True, save_name="mlp_mse")
plot_training_history(mlp_history, title="MLP + MSE")

## 4. 评估与对比

In [ ]:
from evaluate import compare_models

model_configs = [
    {"name": "Linear_MSE", "path": str(CHECKPOINT_DIR / "linear_mse_best.pt"), "is_mlp": False, "n_factors": n_factors},
    {"name": "MLP_MSE", "path": str(CHECKPOINT_DIR / "mlp_mse_best.pt"), "is_mlp": True, "n_factors": n_factors},
]
comparison = compare_models(model_configs, test_ds)